In [7]:
from langgraph.graph import StateGraph, START, END
from typing import TypedDict

In [8]:
class BatsmanState(TypedDict):
    runs: int
    balls: int
    fours: int
    sixes: int

    sr: float
    bpb: float
    boundary_percent: float
    summary: float

In [9]:
def calculate_sr(state: BatsmanState):
    sr = (state['runs']/state['balls'])*100
    return {'sr': sr}

In [10]:
def calculate_bpb(state: BatsmanState):
    bpb = state['balls']/(state['fours'] + state['sixes']) 
    return {'bpb': bpb}

In [11]:
def calculate_boundary_percent(state: BatsmanState):
    boundary_percent = (((state['fours']*4) + (state['sixes']*6))/state['runs'])*100
    return {'boundary_percent': boundary_percent }

In [12]:
def summary(state: BatsmanState):
    summary = f"""
Strike Rate - {state['sr']}\n
Ball Per Boundary - {state['bpb']}\n
Boundary percent - {state['boundary_percent']}
"""
    return {'summary': summary}

In [14]:
graph = StateGraph(BatsmanState)
graph.add_node('calculate_sr', calculate_sr)
graph.add_node('calculate_bpb', calculate_bpb)
graph.add_node('calculate_boundary_percent',calculate_boundary_percent)
graph.add_node('summary', summary)

graph.add_edge(START, 'calculate_sr')
graph.add_edge(START, 'calculate_bpb')
graph.add_edge(START, 'calculate_boundary_percent')
graph.add_edge('calculate_sr', 'summary')
graph.add_edge('calculate_bpb', 'summary')
graph.add_edge('calculate_boundary_percent', 'summary')
graph.add_edge('summary',END)

workflow = graph.compile()


In [15]:
initial_state = {
    'runs': 120,
    'balls': 100,
    'fours': 10,
    'sixes': 5
}
final_state = workflow.invoke(initial_state)

In [16]:
print(final_state)

{'runs': 120, 'balls': 100, 'fours': 10, 'sixes': 5, 'sr': 120.0, 'bpb': 6.666666666666667, 'boundary_percent': 58.333333333333336, 'summary': '\nStrike Rate - 120.0\n\nBall Per Boundary - 6.666666666666667\n\nBoundary percent - 58.333333333333336\n'}
